# Apache Iceberg com Apache Spark (PySpark)

Demonstração de **DDL, INSERT, UPDATE e DELETE** em uma tabela Iceberg.

## Cenário

Tabela única **`carros`**: o cadastro de veículos de uma seguradora. O dataset (`data/carros.csv`, com 8 registros) foi criado pelo grupo a partir do exemplo visto em aula.

### Modelo ER

```mermaid
erDiagram
    CARROS {
        string placa PK
        string modelo
        string chassi
        string marca
        int ano
        string cor
    }
```

| Coluna | Tipo | Descrição |
|---|---|---|
| placa | STRING | Placa do veículo (chave) |
| modelo | STRING | Modelo |
| chassi | STRING | Número do chassi |
| marca | STRING | Fabricante |
| ano | INT | Ano de fabricação |
| cor | STRING | Cor |

## 1. SparkSession com Apache Iceberg

O JAR do Iceberg é baixado do Maven na primeira execução (`spark.jars.packages`). Usamos um **catálogo local do tipo `hadoop`** chamado `local`, que grava tudo na pasta `warehouse/iceberg`.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession

WAREHOUSE = (Path("..") / "warehouse" / "iceberg").resolve()

spark = (
    SparkSession.builder.appName("apache-iceberg")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", str(WAREHOUSE))
    .getOrCreate()
)

# Recria a tabela a cada execução
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.seguradora")
spark.sql("DROP TABLE IF EXISTS local.seguradora.carros PURGE")

spark.version

26/10/03 18:43:36 WARN Utils: Your hostname, NF-ENG-ISABELLEFELTRIN resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/03 18:43:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/isafel/trabalho-apachespark-eng_dados/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/isafel/.ivy2/cache
The jars for the packages stored in: /home/isafel/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-682adeae-bb80-4777-ac92-5ff73afb59b9;1.0
	confs: [default]


	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.9.2/iceberg-spark-runtime-3.5_2.12-1.9.2.jar ...


	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2!iceberg-spark-runtime-3.5_2.12.jar (12353ms)
:: resolution report :: resolve 859ms :: artifacts dl 12357ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.9.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   1   |   1   |   0   ||   1   |   1   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-682adeae-bb80-4777-ac92-5ff73afb59b9
	confs: [default]
	1 artifacts copied, 0 already retrieved (44317kB/41ms)
26/10/03 18:44:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes wh

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


'3.5.9'

## Dados de origem (CSV)

In [2]:
df = spark.read.csv(
    "../data/carros.csv",
    header=True,
    schema="placa STRING, modelo STRING, chassi STRING, marca STRING, ano INT, cor STRING",
)
df.createOrReplaceTempView("carros_csv")
df.show()

+-------+---------+-----------+----------+----+-------+
|  placa|   modelo|     chassi|     marca| ano|    cor|
+-------+---------+-----------+----------+----+-------+
|ALD3834|     CLIO|34574215969|   RENAULT|2011| BRANCO|
|CCR8096|    CRETA|88547875547|   HYUNDAI|2020| BRANCO|
|DLA3438|    PUNTO|98823483434|      FIAT|2013|  PRETO|
|EEE1056|ECO SPORT|56753453455|      FORD|2020|   AZUL|
|FFR1234|    PALIO|32383478747|      FIAT|2020|AMARELO|
|GQY6753|      S10|72004160549|        GM|2015|  PRETO|
|IAC8974|   TIGUAN|77130757746|VOLKSWAGEN|2020|   AZUL|
|JIE0952|   PASSAT|87493270405|VOLKSWAGEN|2016|  CINZA|
+-------+---------+-----------+----------+----+-------+



## 2. DDL: criação da tabela Iceberg

In [3]:
spark.sql("""
    CREATE TABLE local.seguradora.carros (
        placa  STRING,
        modelo STRING,
        chassi STRING,
        marca  STRING,
        ano    INT,
        cor    STRING
    )
    USING iceberg
""")

spark.sql("INSERT INTO local.seguradora.carros SELECT * FROM carros_csv")  # carga inicial
spark.sql("SELECT * FROM local.seguradora.carros ORDER BY placa").show()

+-------+---------+-----------+----------+----+-------+
|  placa|   modelo|     chassi|     marca| ano|    cor|
+-------+---------+-----------+----------+----+-------+
|ALD3834|     CLIO|34574215969|   RENAULT|2011| BRANCO|
|CCR8096|    CRETA|88547875547|   HYUNDAI|2020| BRANCO|
|DLA3438|    PUNTO|98823483434|      FIAT|2013|  PRETO|
|EEE1056|ECO SPORT|56753453455|      FORD|2020|   AZUL|
|FFR1234|    PALIO|32383478747|      FIAT|2020|AMARELO|
|GQY6753|      S10|72004160549|        GM|2015|  PRETO|
|IAC8974|   TIGUAN|77130757746|VOLKSWAGEN|2020|   AZUL|
|JIE0952|   PASSAT|87493270405|VOLKSWAGEN|2016|  CINZA|
+-------+---------+-----------+----------+----+-------+



## 3. INSERT

Inclusão de um carro novo.

In [4]:
spark.sql("""
    INSERT INTO local.seguradora.carros
    VALUES ('KLM2468', 'ONIX', '11223344556', 'GM', 2023, 'PRATA')
""")
spark.sql("SELECT * FROM local.seguradora.carros ORDER BY placa").show()

+-------+---------+-----------+----------+----+-------+
|  placa|   modelo|     chassi|     marca| ano|    cor|
+-------+---------+-----------+----------+----+-------+
|ALD3834|     CLIO|34574215969|   RENAULT|2011| BRANCO|
|CCR8096|    CRETA|88547875547|   HYUNDAI|2020| BRANCO|
|DLA3438|    PUNTO|98823483434|      FIAT|2013|  PRETO|
|EEE1056|ECO SPORT|56753453455|      FORD|2020|   AZUL|
|FFR1234|    PALIO|32383478747|      FIAT|2020|AMARELO|
|GQY6753|      S10|72004160549|        GM|2015|  PRETO|
|IAC8974|   TIGUAN|77130757746|VOLKSWAGEN|2020|   AZUL|
|JIE0952|   PASSAT|87493270405|VOLKSWAGEN|2016|  CINZA|
|KLM2468|     ONIX|11223344556|        GM|2023|  PRATA|
+-------+---------+-----------+----------+----+-------+



## 4. UPDATE

O ECO SPORT (placa `EEE1056`) foi pintado de vermelho.

In [5]:
spark.sql("UPDATE local.seguradora.carros SET cor = 'VERMELHO' WHERE placa = 'EEE1056'")
spark.sql("SELECT * FROM local.seguradora.carros WHERE placa = 'EEE1056'").show()

+-------+---------+-----------+-----+----+--------+
|  placa|   modelo|     chassi|marca| ano|     cor|
+-------+---------+-----------+-----+----+--------+
|EEE1056|ECO SPORT|56753453455| FORD|2020|VERMELHO|
+-------+---------+-----------+-----+----+--------+



## 5. DELETE

Exclusão dos carros fabricados antes de 2015 (CLIO 2011 e PUNTO 2013).

In [6]:
spark.sql("DELETE FROM local.seguradora.carros WHERE ano < 2015")
spark.sql("SELECT * FROM local.seguradora.carros ORDER BY placa").show()

+-------+---------+-----------+----------+----+--------+
|  placa|   modelo|     chassi|     marca| ano|     cor|
+-------+---------+-----------+----------+----+--------+
|CCR8096|    CRETA|88547875547|   HYUNDAI|2020|  BRANCO|
|EEE1056|ECO SPORT|56753453455|      FORD|2020|VERMELHO|
|FFR1234|    PALIO|32383478747|      FIAT|2020| AMARELO|
|GQY6753|      S10|72004160549|        GM|2015|   PRETO|
|IAC8974|   TIGUAN|77130757746|VOLKSWAGEN|2020|    AZUL|
|JIE0952|   PASSAT|87493270405|VOLKSWAGEN|2016|   CINZA|
|KLM2468|     ONIX|11223344556|        GM|2023|   PRATA|
+-------+---------+-----------+----------+----+--------+



## 6. Snapshots e time travel

Cada operação gera um novo **snapshot**, consultável pela tabela de metadados `.snapshots`.

In [7]:
snapshots = spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM local.seguradora.carros.snapshots
    ORDER BY committed_at
""")
snapshots.show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|8582624940116778314|2026-10-03 18:44:06.07 |append   |
|8542463227283315646|2026-10-03 18:44:06.922|append   |
|5042220723700146039|2026-10-03 18:44:08.133|overwrite|
|4638173737208031735|2026-10-03 18:44:08.732|overwrite|
+-------------------+-----------------------+---------+



Consultando a tabela como ela estava **logo após a carga inicial** (primeiro snapshot):

In [8]:
primeiro_snapshot = snapshots.first()["snapshot_id"]
spark.sql(
    f"SELECT * FROM local.seguradora.carros VERSION AS OF {primeiro_snapshot} ORDER BY placa"
).show()

+-------+---------+-----------+----------+----+-------+
|  placa|   modelo|     chassi|     marca| ano|    cor|
+-------+---------+-----------+----------+----+-------+
|ALD3834|     CLIO|34574215969|   RENAULT|2011| BRANCO|
|CCR8096|    CRETA|88547875547|   HYUNDAI|2020| BRANCO|
|DLA3438|    PUNTO|98823483434|      FIAT|2013|  PRETO|
|EEE1056|ECO SPORT|56753453455|      FORD|2020|   AZUL|
|FFR1234|    PALIO|32383478747|      FIAT|2020|AMARELO|
|GQY6753|      S10|72004160549|        GM|2015|  PRETO|
|IAC8974|   TIGUAN|77130757746|VOLKSWAGEN|2020|   AZUL|
|JIE0952|   PASSAT|87493270405|VOLKSWAGEN|2016|  CINZA|
+-------+---------+-----------+----------+----+-------+



## 7. Arquivos gerados

Dados em **Parquet** (`data/`) + metadados em **JSON/Avro** (`metadata/`).

In [9]:
tabela = WAREHOUSE / "seguradora" / "carros"
for arquivo in sorted(tabela.rglob("*")):
    if arquivo.is_file() and not arquivo.name.endswith(".crc"):
        print(arquivo.relative_to(tabela))

data/00000-1-b8c90e1f-7bf7-45f1-adfa-214a5261ccdd-0-00001.parquet
data/00000-13-7cb1d173-3eca-4b35-8bf7-105a1dcab318-0-00001.parquet
data/00000-3-934e9adb-47df-42e9-a53a-24ab5a354499-0-00001.parquet
data/00000-8-414f0df7-9331-463f-81c8-dc22bdfbde4b-0-00001.parquet
metadata/51c870d7-673f-4523-964a-f3e067545d59-m0.avro
metadata/9d91a5a9-92e0-440f-8874-5dd78eaf46cc-m0.avro
metadata/9d91a5a9-92e0-440f-8874-5dd78eaf46cc-m1.avro
metadata/a17acc2f-1cd8-4421-b9be-8a5823992b29-m0.avro
metadata/a17acc2f-1cd8-4421-b9be-8a5823992b29-m1.avro
metadata/d6596dc7-de10-43a7-81ee-6744fc235b1f-m0.avro
metadata/snap-4638173737208031735-1-9d91a5a9-92e0-440f-8874-5dd78eaf46cc.avro
metadata/snap-5042220723700146039-1-a17acc2f-1cd8-4421-b9be-8a5823992b29.avro
metadata/snap-8542463227283315646-1-d6596dc7-de10-43a7-81ee-6744fc235b1f.avro
metadata/snap-8582624940116778314-1-51c870d7-673f-4523-964a-f3e067545d59.avro
metadata/v1.metadata.json
metadata/v2.metadata.json
metadata/v3.metadata.json
metadata/v4.metadata.

In [10]:
spark.stop()